# Gap/Seam Detection Evaluation

This notebook evaluates how well a segmentation model captures the **gaps/seams between stones** rather than the stones themselves.

## Methodology:
The key insight is that morphological closing fills in gaps between stones. Therefore:
- `gaps = closed_mask - original_mask`

By comparing GT gaps against predicted gaps, we get a metric that directly measures stone boundary detection quality.

## Workflow:
1. **Set parameters** (paths, kernel size for closing)
2. **Generate ROI & extract gaps** from ground truth
3. **Extract gaps from prediction** (within the ROI)
4. **Calculate gap-specific metrics** (IoU, precision, recall)
5. **Visualize gap comparison**

---
## Cell 1: Parameters

Set your paths and kernel size here. Re-run subsequent cells after changing these.

In [ ]:
# ============================================================
# PARAMETERS - EDIT THESE
# ============================================================

# Paths to your masks
GT_MASK_PATH = "/path/to/ground_truth_mask.png"  # Ground truth RGB mask
PRED_MASK_PATH = "/path/to/predicted_mask.png"   # Predicted RGB mask

# Output directory for results
OUTPUT_DIR = "/path/to/output/"

# Gap Detection Parameters
# ---------------------------
# Kernel size for morphological closing (in pixels)
# This should be large enough to bridge gaps between stones.
# The same kernel is used for both ROI generation and gap extraction.
# With gaps of 5-10cm and resolution of ~2-6 mm/px, gaps are roughly 8-60 pixels.
# A kernel of 40-60 pixels radius should work well.
KERNEL_RADIUS = 40  # Radius of the disk structuring element

print(f"Ground Truth: {GT_MASK_PATH}")
print(f"Prediction:   {PRED_MASK_PATH}")
print(f"Output Dir:   {OUTPUT_DIR}")
print(f"Kernel Radius: {KERNEL_RADIUS} pixels (diameter: {2*KERNEL_RADIUS + 1})")

---
## Cell 2: Imports and Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from PIL import Image
import pandas as pd
from scipy import ndimage
from skimage.morphology import disk, binary_closing
from typing import Dict, Tuple
import os
import warnings
warnings.filterwarnings('ignore')

Image.MAX_IMAGE_PIXELS = None

# RGB to class mapping
RGB_TO_CLASS = {
    (0, 0, 0): 0,       # Black -> Background
    (0, 0, 255): 1,     # Blue -> Ashlar
    (255, 0, 0): 2,     # Red -> Polygonal
    (255, 255, 0): 3    # Yellow -> Quarry Stone
}

print("Imports complete.")

---
## Cell 3: Helper Functions

In [ ]:
def rgb_to_class_mask(rgb_image: np.ndarray) -> np.ndarray:
    """
    Convert RGB mask to class indices.
    Handles unmapped colors by assigning to nearest class.
    """
    height, width = rgb_image.shape[:2]
    class_mask = np.zeros((height, width), dtype=np.uint8)
    
    for rgb_tuple, class_idx in RGB_TO_CLASS.items():
        color_mask = np.all(rgb_image == rgb_tuple, axis=2)
        class_mask[color_mask] = class_idx
    
    # Handle unmapped pixels (compression artifacts, anti-aliasing)
    mapped_pixels = np.zeros((height, width), dtype=bool)
    for rgb_tuple in RGB_TO_CLASS.keys():
        mapped_pixels |= np.all(rgb_image == rgb_tuple, axis=2)
    
    unmapped_count = np.sum(~mapped_pixels)
    if unmapped_count > 0:
        print(f"  Note: {unmapped_count} pixels with unmapped colors -> mapped to nearest class")
        unmapped_indices = np.where(~mapped_pixels)
        for i in range(len(unmapped_indices[0])):
            y, x = unmapped_indices[0][i], unmapped_indices[1][i]
            pixel_rgb = rgb_image[y, x]
            min_dist = float('inf')
            nearest_class = 0
            for rgb_tuple, class_idx in RGB_TO_CLASS.items():
                dist = np.sqrt(np.sum((pixel_rgb.astype(float) - np.array(rgb_tuple).astype(float))**2))
                if dist < min_dist:
                    min_dist = dist
                    nearest_class = class_idx
            class_mask[y, x] = nearest_class
    
    return class_mask


def extract_gaps_via_closing(
    binary_mask: np.ndarray, 
    kernel_radius: int,
    roi_mask: np.ndarray = None
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Extract gap pixels using morphological closing.
    
    Gaps are defined as pixels that:
    - Are 0 (background/gap) in the original binary mask
    - Become 1 (filled) after morphological closing
    
    Args:
        binary_mask: Binary mask (1=stone, 0=background/gap)
        kernel_radius: Radius of disk structuring element
        roi_mask: Optional ROI to restrict the operation
    
    Returns:
        Tuple of (closed_mask, gap_mask)
    """
    # Apply ROI if provided
    if roi_mask is not None:
        working_mask = binary_mask * roi_mask
    else:
        working_mask = binary_mask
    
    # Create structuring element
    selem = disk(kernel_radius)
    
    # Apply morphological closing
    closed_mask = binary_closing(working_mask.astype(bool), selem)
    
    # Extract gaps: pixels that were 0 but became 1 after closing
    gap_mask = closed_mask & ~working_mask.astype(bool)
    
    # If ROI was provided, ensure gaps are within ROI
    if roi_mask is not None:
        gap_mask = gap_mask & roi_mask
    
    return closed_mask, gap_mask


def calculate_gap_metrics(
    gt_gaps: np.ndarray, 
    pred_gaps: np.ndarray
) -> Dict[str, float]:
    """
    Calculate IoU, precision, recall, and F1 for gap detection.
    
    Args:
        gt_gaps: Binary mask of ground truth gaps
        pred_gaps: Binary mask of predicted gaps
    
    Returns:
        Dictionary with metrics
    """
    gt_gaps = gt_gaps.astype(bool)
    pred_gaps = pred_gaps.astype(bool)
    
    # Basic counts
    gt_gap_pixels = np.sum(gt_gaps)
    pred_gap_pixels = np.sum(pred_gaps)
    
    # Intersection and union
    intersection = np.sum(gt_gaps & pred_gaps)
    union = np.sum(gt_gaps | pred_gaps)
    
    # IoU
    iou = intersection / union if union > 0 else 0.0
    
    # Precision: Of predicted gaps, how many are correct?
    precision = intersection / pred_gap_pixels if pred_gap_pixels > 0 else 0.0
    
    # Recall: Of GT gaps, how many were found?
    recall = intersection / gt_gap_pixels if gt_gap_pixels > 0 else 0.0
    
    # F1 Score
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    
    return {
        'gap_iou': iou,
        'gap_precision': precision,
        'gap_recall': recall,
        'gap_f1': f1,
        'gt_gap_pixels': gt_gap_pixels,
        'pred_gap_pixels': pred_gap_pixels,
        'intersection_pixels': intersection,
        'union_pixels': union
    }


print("Helper functions defined.")

---
## Cell 4: Load Masks and Generate ROI + GT Gaps

This cell loads your masks and extracts the ground truth gaps using morphological closing.

In [ ]:
# Load ground truth
print("Loading ground truth mask...")
gt_rgb = np.array(Image.open(GT_MASK_PATH).convert('RGB'))
gt_mask = rgb_to_class_mask(gt_rgb)
print(f"  Shape: {gt_mask.shape}")
print(f"  Classes present: {np.unique(gt_mask)}")

# Load prediction
print("\nLoading predicted mask...")
pred_rgb = np.array(Image.open(PRED_MASK_PATH).convert('RGB'))
pred_mask = rgb_to_class_mask(pred_rgb)
print(f"  Shape: {pred_mask.shape}")
print(f"  Classes present: {np.unique(pred_mask)}")

# Verify shapes match
assert gt_mask.shape == pred_mask.shape, f"Shape mismatch: GT {gt_mask.shape} vs Pred {pred_mask.shape}"

# Create binary masks (all stones -> 1, background -> 0)
gt_binary = (gt_mask > 0).astype(np.uint8)
pred_binary = (pred_mask > 0).astype(np.uint8)

print(f"\n--- Ground Truth Binary ---")
print(f"  Stone pixels: {np.sum(gt_binary):,}")
print(f"  Background pixels: {np.sum(gt_binary == 0):,}")

# Generate ROI and GT gaps from ground truth
print(f"\nExtracting GT gaps with kernel radius = {KERNEL_RADIUS}...")
gt_closed, gt_gaps = extract_gaps_via_closing(gt_binary, KERNEL_RADIUS)

# The GT closed mask serves as our ROI
roi_mask = gt_closed.astype(bool)

print(f"  ROI pixels (GT closed): {np.sum(roi_mask):,}")
print(f"  GT gap pixels: {np.sum(gt_gaps):,}")

print("\n✓ Ground truth processed. Proceed to extract prediction gaps.")

---
## Cell 5: Extract Prediction Gaps (Within ROI)

Apply the same closing operation to the prediction, but restricted to the ROI.

In [ ]:
# Extract gaps from prediction within ROI
print(f"Extracting prediction gaps within ROI...")

# Apply ROI to prediction before closing
pred_closed, pred_gaps = extract_gaps_via_closing(pred_binary, KERNEL_RADIUS, roi_mask)

print(f"  Prediction stone pixels (within ROI): {np.sum(pred_binary * roi_mask):,}")
print(f"  Prediction gap pixels: {np.sum(pred_gaps):,}")

print("\n✓ Prediction gaps extracted. Proceed to calculate metrics.")

---
## Cell 6: Calculate Gap Metrics

In [ ]:
# Calculate gap-specific metrics
print("Calculating gap detection metrics...\n")

metrics = calculate_gap_metrics(gt_gaps, pred_gaps)

# Print results
print("="*60)
print("GAP DETECTION EVALUATION RESULTS")
print("="*60)

print(f"\nKernel radius: {KERNEL_RADIUS} pixels")
print(f"\nPixel Counts:")
print("-"*40)
print(f"  GT gap pixels:           {metrics['gt_gap_pixels']:,}")
print(f"  Predicted gap pixels:    {metrics['pred_gap_pixels']:,}")
print(f"  Intersection:            {metrics['intersection_pixels']:,}")
print(f"  Union:                   {metrics['union_pixels']:,}")

print(f"\nGap Detection Metrics:")
print("-"*40)
print(f"  Gap IoU:        {metrics['gap_iou']:.4f}")
print(f"  Gap Precision:  {metrics['gap_precision']:.4f}")
print(f"  Gap Recall:     {metrics['gap_recall']:.4f}")
print(f"  Gap F1-Score:   {metrics['gap_f1']:.4f}")
print("="*60)

# Interpretation
print("\nInterpretation:")
if metrics['pred_gap_pixels'] > metrics['gt_gap_pixels'] * 1.2:
    print("  → Model tends to OVER-SEGMENT (more boundaries than GT)")
elif metrics['pred_gap_pixels'] < metrics['gt_gap_pixels'] * 0.8:
    print("  → Model tends to UNDER-SEGMENT (fewer boundaries than GT)")
else:
    print("  → Model produces similar number of boundaries as GT")

if metrics['gap_precision'] > metrics['gap_recall']:
    print("  → Higher precision: predicted gaps are mostly correct, but some GT gaps missed")
elif metrics['gap_recall'] > metrics['gap_precision']:
    print("  → Higher recall: most GT gaps found, but some false positive gaps")
else:
    print("  → Balanced precision and recall")

---
## Cell 7: Visualize Gap Comparison

In [ ]:
# Visualization of gap comparison
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Row 1: Binary masks and their closed versions
axes[0, 0].imshow(gt_binary, cmap='gray')
axes[0, 0].set_title('GT Binary (stones=white)', fontsize=12, fontweight='bold')
axes[0, 0].axis('off')

axes[0, 1].imshow(gt_closed, cmap='gray')
axes[0, 1].set_title(f'GT Closed (kernel r={KERNEL_RADIUS})', fontsize=12, fontweight='bold')
axes[0, 1].axis('off')

axes[0, 2].imshow(gt_gaps, cmap='Greens')
axes[0, 2].set_title(f'GT Gaps ({np.sum(gt_gaps):,} px)', fontsize=12, fontweight='bold')
axes[0, 2].axis('off')

# Row 2: Prediction and comparison
pred_binary_in_roi = pred_binary * roi_mask
axes[1, 0].imshow(pred_binary_in_roi, cmap='gray')
axes[1, 0].set_title('Prediction Binary (in ROI)', fontsize=12, fontweight='bold')
axes[1, 0].axis('off')

axes[1, 1].imshow(pred_closed, cmap='gray')
axes[1, 1].set_title(f'Prediction Closed', fontsize=12, fontweight='bold')
axes[1, 1].axis('off')

axes[1, 2].imshow(pred_gaps, cmap='Blues')
axes[1, 2].set_title(f'Predicted Gaps ({np.sum(pred_gaps):,} px)', fontsize=12, fontweight='bold')
axes[1, 2].axis('off')

plt.tight_layout()
plt.show()

---
## Cell 8: Gap Overlap Visualization

Shows where GT and predicted gaps match (green), where GT gaps were missed (red), and where false positive gaps appear (blue).

In [ ]:
# Create overlap visualization
# Green = correct (intersection)
# Red = missed GT gaps (GT only)
# Blue = false positive gaps (Pred only)

gt_gaps_bool = gt_gaps.astype(bool)
pred_gaps_bool = pred_gaps.astype(bool)

# Classification
correct_gaps = gt_gaps_bool & pred_gaps_bool      # True positive
missed_gaps = gt_gaps_bool & ~pred_gaps_bool      # False negative
false_positive_gaps = ~gt_gaps_bool & pred_gaps_bool  # False positive

# Create RGB visualization
overlap_viz = np.zeros((*gt_gaps.shape, 3), dtype=np.uint8)
overlap_viz[correct_gaps] = [0, 255, 0]        # Green
overlap_viz[missed_gaps] = [255, 0, 0]         # Red
overlap_viz[false_positive_gaps] = [0, 0, 255] # Blue

# Also show stone outlines for context
gt_stone_boundary = gt_binary.astype(float) - ndimage.binary_erosion(gt_binary).astype(float)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Left: Gap overlap
axes[0].imshow(overlap_viz)
axes[0].set_title('Gap Detection Comparison\nGreen=Correct, Red=Missed, Blue=False Positive', 
                  fontsize=12, fontweight='bold')
axes[0].axis('off')

# Right: With stone context
# Show GT stones as gray background
context_viz = np.stack([gt_binary * 128] * 3, axis=-1).astype(np.uint8)
context_viz[correct_gaps] = [0, 255, 0]
context_viz[missed_gaps] = [255, 0, 0]
context_viz[false_positive_gaps] = [0, 0, 255]

axes[1].imshow(context_viz)
axes[1].set_title('Gap Comparison with Stone Context\n(Gray=GT stones)', 
                  fontsize=12, fontweight='bold')
axes[1].axis('off')

plt.tight_layout()

# Print statistics
print("\nGap Classification Statistics:")
print("-"*40)
print(f"  Correct gaps (TP):       {np.sum(correct_gaps):,} pixels")
print(f"  Missed gaps (FN):        {np.sum(missed_gaps):,} pixels")
print(f"  False positive gaps (FP): {np.sum(false_positive_gaps):,} pixels")

plt.show()

---
## Cell 9: Save Results

In [ ]:
# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Build report dataframe
report_data = {
    'Metric': ['Gap IoU', 'Gap Precision', 'Gap Recall', 'Gap F1-Score',
               'GT Gap Pixels', 'Predicted Gap Pixels', 'Intersection', 'Union'],
    'Value': [metrics['gap_iou'], metrics['gap_precision'], metrics['gap_recall'], metrics['gap_f1'],
              metrics['gt_gap_pixels'], metrics['pred_gap_pixels'], 
              metrics['intersection_pixels'], metrics['union_pixels']]
}
report_df = pd.DataFrame(report_data)

print("\nGap Detection Report:")
print(report_df.to_string(index=False))

# Save report
report_path = os.path.join(OUTPUT_DIR, 'gap_evaluation_report.csv')
report_df.to_csv(report_path, index=False)
print(f"\n✓ Report saved to: {report_path}")

# Save gap masks for reference
gt_gaps_path = os.path.join(OUTPUT_DIR, 'gt_gaps.png')
Image.fromarray((gt_gaps * 255).astype(np.uint8)).save(gt_gaps_path)
print(f"✓ GT gaps mask saved to: {gt_gaps_path}")

pred_gaps_path = os.path.join(OUTPUT_DIR, 'pred_gaps.png')
Image.fromarray((pred_gaps * 255).astype(np.uint8)).save(pred_gaps_path)
print(f"✓ Predicted gaps mask saved to: {pred_gaps_path}")

# Save overlap visualization
overlap_path = os.path.join(OUTPUT_DIR, 'gap_comparison_overlay.png')
Image.fromarray(overlap_viz).save(overlap_path)
print(f"✓ Gap comparison overlay saved to: {overlap_path}")

---
## Cell 10: Summary Bar Chart

In [ ]:
# Gap metrics bar chart
fig, ax = plt.subplots(figsize=(10, 6))

metric_names = ['IoU', 'Precision', 'Recall', 'F1-Score']
metric_values = [metrics['gap_iou'], metrics['gap_precision'], 
                 metrics['gap_recall'], metrics['gap_f1']]
colors = ['#2ecc71', '#3498db', '#e74c3c', '#9b59b6']

bars = ax.bar(metric_names, metric_values, color=colors, edgecolor='black', linewidth=1.5)

# Add value labels
for bar, score in zip(bars, metric_values):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
            f'{score:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=12)

ax.set_ylabel('Score', fontsize=12)
ax.set_title(f'Gap Detection Metrics (Kernel r={KERNEL_RADIUS})', fontsize=14, fontweight='bold')
ax.set_ylim(0, 1.15)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()

# Save
chart_path = os.path.join(OUTPUT_DIR, 'gap_metrics_chart.png')
plt.savefig(chart_path, dpi=150, bbox_inches='tight')
print(f"✓ Gap metrics chart saved to: {chart_path}")

plt.show()